# scratch

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

rng = np.random.default_rng()
π = np.pi

%matplotlib ipympl
    
%load_ext autoreload
%autoreload all

In [ ]:
fig = plt.figure(constrained_layout=True)

def mkax():
    fig.clf()
    return fig.gca()

In [ ]:
ax = mkax()
ax.plot(signal.get_window("hann", 50))
ax.plot(signal.get_window("hamming", 50))
ax.plot(signal.windows.kaiser(50, 5))

In [ ]:
from modreczoo.data import load_dataset_loader

dl, x, meta, label_to_id = load_dataset_loader(
    Path("data/cspb_2018r2/"), model_name="resnet_1d", channel_format="real_imag")

In [ ]:
n = np.arange(x.shape[1])
i = 1

fig.clf()
ax, sax = fig.subplots(2, 1)
ax.plot(n, x[i].real, n, x[i].imag)
ax.set(title=f"{meta[i,"modulation"]}, OSR={meta[i,"osr"]}, SNR={meta[i,"snr_db"]:.1f}dB");

In [ ]:
meta[i]

## Understanding SRRC

In [ ]:
ax.scatter?

In [ ]:
1/(sps*up)

In [ ]:
span / hf.size

In [ ]:
span*up*2+1

In [ ]:
hf.size

In [ ]:
from modreczoo.simulation import srrc_filter
from numpy.fft import fft, fftfreq, fftshift

sps = 10
betas = [0.01, 0.25, 0.5, 0.75, 1.0]
span = 100

fig.clf()
tax, fax = fig.subplots(2, 1)
for beta in betas:
    # h = srrc_filter(sps, beta, span)
    # τ = np.linspace(-span/2, span/2, h.size)
    hf = srrc_filter(sps, beta, span)
    τf = np.linspace(-span/2, span/2, hf.size)

    Hf = fftshift(fft(hf))
    f = fftshift(fftfreq(hf.size))

    tax.plot(τf, hf, '-', label=f"$\\beta={beta}$")
    fax.plot(f[f>0], 20*np.log10(np.abs(Hf[f>0])))

fax.vlines(np.minimum(1/(2*sps), 0.5)*np.r_[1, 1+betas[-1]], *fax.get_ylim(), 'k', '--')
tax.axhline(0, color='k', ls='--')
tax.legend()

## Understanding Entropy

In [ ]:
from scipy import stats

def cross_entropy(P, Q, x, nats=True):
    scale = 1.0 if nats else 1.0/np.log(2)
    return scale * np.trapezoid(-P*np.log(Q), x)

def entropy(P, x, nats=True):
    return cross_entropy(P, P, x, nats)

# def joint_entropy(P, Q, x, nats=True):
#     return np.trapezoid(

def kl_divergence(P, Q, x, nats=True):
    scale = 1.0 if nats else 1.0/np.log(2)
    return scale * np.trapezoid(P*np.log(P/Q), x)

relative_entropy = kl_divergence

P = stats.norm(loc=0, scale=1)
# Q = stats.norm(loc=2, scale=1)
# Q = stats.uniform()
Q = stats.norm(loc=10, scale=1)
x = np.linspace(-10, 20, 1000)
dx = x[1] - x[0]

nats = False
H_PQ = cross_entropy(P.pdf(x), Q.pdf(x), x, nats)
H_QP = cross_entropy(Q.pdf(x), P.pdf(x), x, nats)
H_P = entropy(P.pdf(x), x, nats)
H_Q = entropy(Q.pdf(x), x, nats)
D_PQ = kl_divergence(P.pdf(x), Q.pdf(x), x, nats)
D_QP = kl_divergence(Q.pdf(x), P.pdf(x), x, nats)

ax = mkax()
ax.plot(x, P.pdf(x), x, Q.pdf(x))
ax.set(title=f"{H_P=:.2f}, {H_Q=:.2f}\n{H_PQ=:.2f}, {H_QP=:.2f}\n{D_PQ=:.2f}, {D_QP=:.2f}")

np.allclose(D_PQ, H_PQ - H_P), np.allclose(D_QP, H_QP - H_Q)

## Spectrogram Explorer

In [ ]:
from importlib import reload
import modreczoo.plotting
reload(modreczoo.plotting)

from modreczoo.plotting import interactive_iq_explorer
explorer = interactive_iq_explorer("data/cspb_2018r2/", width=1200)

In [ ]:
import polars as pl

df = pl.read_parquet("data/cspb_2018r2/delivered/signal_record_C_2023.parquet")
df[16000-10:16000+10]

In [ ]:
ax = mkax()
H, *_, im = ax.hist2d(df["upsample_factor"], df["base_symbol_period"], 
                      (range(1, df["upsample_factor"].max()+2), 
                      range(1, int(df["base_symbol_period"].max()+2))))
fig.colorbar(im)
# ax.hist(meta["upsample_factor"] * meta["base_symbol_period"])